<a href="https://colab.research.google.com/github/adanzee/Neuron-regularization/blob/main/model_neuron_regularization_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:


import os, time, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.datasets import fashion_mnist
from sklearn.metrics import confusion_matrix

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = ['T-shirt/top','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']


print("\n[1] Loading Fashion-MNIST …")
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train = x_train.astype('float32') / 255.0   # normalise to [0,1]
x_test  = x_test.astype('float32')  / 255.0
x_train_flat = x_train.reshape(-1, 784)         # flatten 28×28 → 784
x_test_flat  = x_test.reshape(-1, 784)
print(f"  Train: {x_train_flat.shape}  Test: {x_test_flat.shape}")


def build_baseline(neurons=(64,32), activation='relu', l2_lam=None):
    reg = regularizers.l2(l2_lam) if l2_lam else None
    return models.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(neurons[0], activation=activation, kernel_regularizer=reg),
        layers.Dense(neurons[1], activation=activation, kernel_regularizer=reg),
        layers.Dense(10, activation='softmax')
    ])

def compile_and_train(model, epochs=15, batch_size=32, lr=0.001, val=0.2):
    model.compile(optimizer=optimizers.Adam(lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model.fit(x_train_flat, y_train, epochs=epochs,
                     batch_size=batch_size, validation_split=val, verbose=0)

def plot_curves(histories, labels, title, fname):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = plt.cm.tab10(np.linspace(0, 1, len(histories)))
    for h, lbl, c in zip(histories, labels, colors):
        axes[0].plot(h.history['accuracy'],     color=c, label=f'{lbl} train')
        axes[0].plot(h.history['val_accuracy'], color=c, ls='--', label=f'{lbl} val')
        axes[1].plot(h.history['loss'],         color=c, label=f'{lbl} train')
        axes[1].plot(h.history['val_loss'],     color=c, ls='--', label=f'{lbl} val')
    for ax, m in zip(axes, ['Accuracy','Loss']):
        ax.set(title=f'{title} – {m}', xlabel='Epoch', ylabel=m)
        ax.legend(fontsize=7); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{fname}", dpi=150, bbox_inches='tight'); plt.close()

print("\n[2] Training base model and 3 architecture variations …")
arch_cfgs = [((64,32),'Base (64-32)'),((128,64),'Var1 (128-64)'),
             ((32,16),'Var2 (32-16)'),((128,128),'Var3 (128-128)')]
arch_hist, arch_lbl, arch_scores = [], [], []
for neurons, lbl in arch_cfgs:
    m = build_baseline(neurons)
    h = compile_and_train(m)
    loss, acc = m.evaluate(x_test_flat, y_test, verbose=0)
    arch_hist.append(h); arch_lbl.append(lbl)
    arch_scores.append((lbl, acc, loss))
    print(f"  {lbl:20s}  acc={acc:.4f}  loss={loss:.4f}")

plot_curves(arch_hist, arch_lbl, 'Architecture Variations', 'task3_arch_curves.png')

# Base model confusion matrix + sample predictions
base_m = build_baseline(); compile_and_train(base_m)
y_pred = np.argmax(base_m.predict(x_test_flat, verbose=0), axis=1)

fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set(title='Confusion Matrix – Base Model', xlabel='Predicted', ylabel='True')
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/task3_confusion.png", dpi=150, bbox_inches='tight'); plt.close()

fig, axes = plt.subplots(2, 5, figsize=(14,6))
for ax, idx in zip(axes.flat, np.random.choice(len(x_test), 10, replace=False)):
    ax.imshow(x_test[idx], cmap='gray')
    p, t = CLASS_NAMES[y_pred[idx]], CLASS_NAMES[y_test[idx]]
    ax.set_title(f'P:{p}\nT:{t}', color='green' if p==t else 'red', fontsize=7)
    ax.axis('off')
plt.suptitle('10 Sample Predictions'); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/task3_samples.png", dpi=150, bbox_inches='tight'); plt.close()


print("\n[3] Training regularization models A-D …")

def model_a():  # Dropout
    return models.Sequential([layers.Input((784,)),
        layers.Dense(64,'relu'), layers.Dropout(0.3),
        layers.Dense(32,'relu'), layers.Dropout(0.3),
        layers.Dense(10,'softmax')])

def model_b():  # L2
    r = regularizers.l2(0.001)
    return models.Sequential([layers.Input((784,)),
        layers.Dense(64,'relu', kernel_regularizer=r),
        layers.Dense(32,'relu', kernel_regularizer=r),
        layers.Dense(10,'softmax')])

def model_c():  # BatchNorm
    return models.Sequential([layers.Input((784,)),
        layers.Dense(64,'relu'), layers.BatchNormalization(),
        layers.Dense(32,'relu'), layers.BatchNormalization(),
        layers.Dense(10,'softmax')])

def model_d():  # Combined
    r = regularizers.l2(0.0005)
    return models.Sequential([layers.Input((784,)),
        layers.Dense(64,'relu', kernel_regularizer=r), layers.BatchNormalization(), layers.Dropout(0.2),
        layers.Dense(32,'relu', kernel_regularizer=r), layers.BatchNormalization(), layers.Dropout(0.2),
        layers.Dense(10,'softmax')])

reg_builders = [build_baseline, model_a, model_b, model_c, model_d]
reg_names    = ['Baseline','Dropout','L2','BatchNorm','Combined']
reg_hist, reg_scores = [], []
for builder, name in zip(reg_builders, reg_names):
    m = builder(); h = compile_and_train(m)
    loss, acc = m.evaluate(x_test_flat, y_test, verbose=0)
    reg_hist.append(h); reg_scores.append((name, acc, loss))
    print(f"  {name:12s}  acc={acc:.4f}  loss={loss:.4f}")

plot_curves(reg_hist, reg_names, 'Regularization Models', 'task5_reg_curves.png')

# Weight distribution histograms
bl = build_baseline(); compile_and_train(bl)
l2 = model_b();        compile_and_train(l2)
dr = model_a();        compile_and_train(dr)
fig, axes = plt.subplots(1, 3, figsize=(15,5))
for ax, m, name in zip(axes, [bl, l2, dr], ['Baseline','L2','Dropout']):
    w = m.layers[0].get_weights()[0].flatten()
    ax.hist(w, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(0, color='red', ls='--'); ax.set_title(f'{name} – Layer 1 Weights')
    ax.set(xlabel='Weight Value', ylabel='Count'); ax.grid(alpha=0.3)
plt.suptitle('Weight Distributions'); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/task5_weight_dists.png", dpi=150, bbox_inches='tight'); plt.close()

# Best model confusion matrix + error analysis
best_idx = np.argmax([s[1] for s in reg_scores])
best_name = reg_scores[best_idx][0]
print(f"  Best regularization model: {best_name}")
best_m = reg_builders[best_idx](); compile_and_train(best_m)
y_pred_best = np.argmax(best_m.predict(x_test_flat, verbose=0), axis=1)

cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set(title=f'Confusion Matrix – {best_name}', xlabel='Predicted', ylabel='True')
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/task5_best_cm.png", dpi=150, bbox_inches='tight'); plt.close()

# Feature visualisation (16 filters)
w_feat = bl.layers[0].get_weights()[0]  # (784, 64)
fig, axes = plt.subplots(4, 4, figsize=(10,10))
for i, ax in enumerate(axes.flat):
    ax.imshow(w_feat[:,i].reshape(28,28), cmap='RdBu_r'); ax.axis('off')
    ax.set_title(f'Filter {i+1}', fontsize=8)
plt.suptitle('16 Feature Detectors – Layer 1'); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/task5_features.png", dpi=150, bbox_inches='tight'); plt.close()

# Misclassified examples
wrong = np.where(y_pred_best != y_test)[0]
fig, axes = plt.subplots(1, 5, figsize=(14,4))
for ax, idx in zip(axes, wrong[:5]):
    ax.imshow(x_test[idx], cmap='gray')
    ax.set_title(f'T:{CLASS_NAMES[y_test[idx]]}\nP:{CLASS_NAMES[y_pred_best[idx]]}',
                 color='red', fontsize=7); ax.axis('off')
plt.suptitle('5 Misclassified Examples'); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/task5_errors.png", dpi=150, bbox_inches='tight'); plt.close()


print("\n[4] Learning rate sweep …")
lrs = [0.1, 0.01, 0.001, 0.0001, 0.00001]
lr_hist, lr_scores = [], []
for lr in lrs:
    m = build_baseline(); h = compile_and_train(m, lr=lr)
    _, acc = m.evaluate(x_test_flat, y_test, verbose=0)
    lr_hist.append(h); lr_scores.append(acc)
    print(f"  lr={lr:.5f}  acc={acc:.4f}")
plot_curves(lr_hist, [str(lr) for lr in lrs], 'Learning Rate', 'task6_lr_curves.png')

print("\n[5] Batch size sweep …")
bsizes = [16, 32, 64, 128, 256]
bs_hist, bs_scores, bs_times = [], [], []
for bs in bsizes:
    m = build_baseline(); t0 = time.time()
    h = compile_and_train(m, batch_size=bs)
    elapsed = time.time() - t0
    _, acc = m.evaluate(x_test_flat, y_test, verbose=0)
    bs_hist.append(h); bs_scores.append(acc); bs_times.append(elapsed)
    print(f"  batch={bs:4d}  acc={acc:.4f}  time={elapsed:.1f}s")
plot_curves(bs_hist, [str(bs) for bs in bsizes], 'Batch Size', 'task6_bs_curves.png')

print("\n[6] Activation function sweep …")
def build_act(act):
    if act == 'leaky_relu':
        return models.Sequential([layers.Input((784,)),
            layers.Dense(64), layers.LeakyReLU(0.1),
            layers.Dense(32), layers.LeakyReLU(0.1),
            layers.Dense(10,'softmax')])
    return models.Sequential([layers.Input((784,)),
        layers.Dense(64, activation=act), layers.Dense(32, activation=act),
        layers.Dense(10,'softmax')])

acts = ['relu','leaky_relu','elu','tanh','swish']
act_hist, act_scores = [], []
for act in acts:
    m = build_act(act); h = compile_and_train(m)
    _, acc = m.evaluate(x_test_flat, y_test, verbose=0)
    act_hist.append(h); act_scores.append(acc)
    print(f"  activation={act:12s}  acc={acc:.4f}")
plot_curves(act_hist, acts, 'Activation Functions', 'task6_act_curves.png')

# Summary bar chart
fig, axes = plt.subplots(1, 3, figsize=(16,5))
axes[0].bar([str(lr) for lr in lrs], lr_scores, color=plt.cm.viridis(np.linspace(0.2,0.8,5)))
axes[0].set(title='Learning Rate vs Accuracy', xlabel='LR', ylabel='Accuracy')
axes[0].tick_params(axis='x', rotation=30); axes[0].grid(alpha=0.3, axis='y')
axes[1].bar([str(bs) for bs in bsizes], bs_scores, color=plt.cm.plasma(np.linspace(0.2,0.8,5)))
axes[1].set(title='Batch Size vs Accuracy', xlabel='Batch Size', ylabel='Accuracy')
axes[1].grid(alpha=0.3, axis='y')
axes[2].bar(acts, act_scores, color=plt.cm.cool(np.linspace(0.2,0.8,5)))
axes[2].set(title='Activation vs Accuracy', xlabel='Activation', ylabel='Accuracy')
axes[2].grid(alpha=0.3, axis='y')
plt.suptitle('Hyperparameter Comparison', fontsize=14); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/task6_hp_summary.png", dpi=150, bbox_inches='tight'); plt.close()

print("\n[7] Generating saliency maps …")
fig, axes = plt.subplots(3, 5, figsize=(16,9))
for col, idx in enumerate(np.random.choice(len(x_test), 5, replace=False)):
    img_t = tf.Variable(x_test_flat[idx:idx+1], dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(img_t)
        pred = bl(img_t)
        top  = tf.argmax(pred[0]).numpy()
        score = pred[0][top]
    grads = tf.abs(tape.gradient(score, img_t)).numpy()[0].reshape(28,28)
    sal_norm = (grads - grads.min()) / (grads.max() - grads.min() + 1e-8)
    t = y_test[idx]
    axes[0,col].imshow(x_test[idx], cmap='gray'); axes[0,col].set_title(CLASS_NAMES[t], fontsize=7); axes[0,col].axis('off')
    axes[1,col].imshow(grads, cmap='hot');         axes[1,col].set_title(CLASS_NAMES[top], fontsize=7); axes[1,col].axis('off')
    axes[2,col].imshow(x_test[idx], cmap='gray', alpha=0.6)
    axes[2,col].imshow(sal_norm, cmap='jet', alpha=0.5)
    axes[2,col].set_title('✓' if top==t else '✗', color='green' if top==t else 'red', fontsize=10)
    axes[2,col].axis('off')
for ax, lbl in zip(axes[:,0], ['Input','Saliency','Overlay']):
    ax.set_ylabel(lbl, fontsize=9)
plt.suptitle('Saliency Maps – 5 Test Samples', fontsize=13); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/task6_saliency.png", dpi=150, bbox_inches='tight'); plt.close()


print("\n" + "="*65)
print("  ANALYSIS REPORT")
print("="*65)

tr_acc = arch_hist[0].history['accuracy'][-1]
va_acc = arch_hist[0].history['val_accuracy'][-1]
gap    = tr_acc - va_acc
print(f"\n── Task 3: Architecture Comparison ──────────────────────────")
for name, acc, loss in arch_scores:
    print(f"  {name:20s}  acc={acc:.4f}  loss={loss:.4f}")
print(f"\n  Generalization gap (base): {gap:.4f} → {'overfitting' if gap>0.05 else 'good fit'}")

print(f"\n── Task 5: Regularization ────────────────────────────────────")
for name, acc, loss in reg_scores:
    print(f"  {name:12s}  acc={acc:.4f}  loss={loss:.4f}")

print(f"\n── Task 6: Best Hyperparameters ──────────────────────────────")
print(f"  Best LR:         {lrs[np.argmax(lr_scores)]}")
print(f"  Best Batch Size: {bsizes[np.argmax(bs_scores)]}")
print(f"  Best Activation: {acts[np.argmax(act_scores)]}")

off_diag = confusion_matrix(y_test, y_pred_best).copy()
np.fill_diagonal(off_diag, 0)
top3 = []
for _ in range(3):
    i, j = np.unravel_index(np.argmax(off_diag), off_diag.shape)
    top3.append((CLASS_NAMES[i], CLASS_NAMES[j], off_diag[i,j]))
    off_diag[i,j] = 0
print(f"\n  Top 3 confused pairs ({best_name}):")
for tc, pc, n in top3:
    print(f"    {tc:15s} → {pc:15s}  ({n} errors)")

print(f"\n  All plots saved to: {OUTPUT_DIR}/")
print("="*65)


[1] Loading Fashion-MNIST …
  Train: (60000, 784)  Test: (10000, 784)

[2] Training base model and 3 architecture variations …
  Base (64-32)          acc=0.8751  loss=0.3749
  Var1 (128-64)         acc=0.8807  loss=0.3930
  Var2 (32-16)          acc=0.8612  loss=0.3876
  Var3 (128-128)        acc=0.8854  loss=0.3808

[3] Training regularization models A-D …
  Baseline      acc=0.8789  loss=0.3595
  Dropout       acc=0.8611  loss=0.3859
  L2            acc=0.8607  loss=0.4574
  BatchNorm     acc=0.8751  loss=0.3683
  Combined      acc=0.8461  loss=0.4915
  Best regularization model: Baseline

[4] Learning rate sweep …
  lr=0.10000  acc=0.1000
  lr=0.01000  acc=0.8502
  lr=0.00100  acc=0.8748
  lr=0.00010  acc=0.8607
  lr=0.00001  acc=0.8181

[5] Batch size sweep …
  batch=  16  acc=0.8715  time=128.7s
  batch=  32  acc=0.8802  time=67.3s
  batch=  64  acc=0.8782  time=38.5s
  batch= 128  acc=0.8733  time=20.6s
  batch= 256  acc=0.8663  time=13.7s

[6] Activation function sweep …
  act